# RoadGuard AI — Multi-Model Experimentation, Benchmarking & Export

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tahadeab-ai/roadguard-ai/blob/main/ai/colab/03_model_experiments_and_evaluation.ipynb)

> **Objective**: Execute comparative evaluation of edge candidate models (YOLOv8n vs YOLOv11n vs optional YOLOv8s ceiling), optimize for safety-critical $F_2$-score, and export INT8 quantized ONNX.
>
> **Execution Hardware**: Google Colab Cloud GPU (NVIDIA T4 / A100). **Zero local GPU computation.**
>
> **Target Taxonomy**:
- `0: D00_longitudinal_crack` (RDD2022 D00)
- `1: D10_transverse_crack` (RDD2022 D10)
- `2: D20_alligator_crack` (RDD2022 D20)
- `3: D40_pothole` (RDD2022 D40)

### Licensing & Risk Notice:
- **Ultralytics**: AGPL-3.0 copyleft license. Network interaction mandates full source disclosure unless an enterprise commercial license is purchased.
- **RDD2022**: CC BY-SA 4.0 (attribution required to Sekimoto Lab).


## 1. Hardware Inspection, Drive Mount & Reproducibility Setup

In [ ]:
import os
import sys
import time
import json
import hashlib
import shutil
from pathlib import Path
import torch
import numpy as np

# Enforce deterministic seed controls
torch.manual_seed(42)
np.random.seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"[SUCCESS] Cloud GPU: {gpu_name} ({vram_gb:.2f} GB VRAM)")
else:
    print("[WARNING] No CUDA GPU detected. Running on CPU is strictly for smoke-testing (1-2 epochs)!")

# Mount Google Drive to persist weights and experiment logs
try:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_EXPERIMENT_DIR = Path('/content/drive/MyDrive/RoadGuard_Models/experiments')
    DRIVE_EXPERIMENT_DIR.mkdir(parents=True, exist_ok=True)
    print(f"Google Drive Experiment Directory: {DRIVE_EXPERIMENT_DIR}")
except Exception as e:
    print(f"Google Drive mount skipped: {e}")
    DRIVE_EXPERIMENT_DIR = Path('/content/roadguard_experiments')
    DRIVE_EXPERIMENT_DIR.mkdir(parents=True, exist_ok=True)

## 2. Install Pinned Dependencies

In [ ]:
# Pin versions for absolute reproducibility
!pip install -q "ultralytics==8.2.32" "torch==2.3.1" "onnx==1.16.1" "onnxruntime==1.18.0" "onnxslim==0.1.31" pyyaml matplotlib

import yaml
import ultralytics
from ultralytics import YOLO
import onnx
import onnxruntime as ort
import cv2

print(f"Ultralytics Version: {ultralytics.__version__}")
print(f"ONNX Version:        {onnx.__version__}")
print(f"ONNX Runtime:        {ort.__version__}")

## 3. Dataset Integrity & Checksum Verification

In [ ]:
DATA_YAML_PATH = Path('/content/roadguard_rdd2022/data.yaml')

if not DATA_YAML_PATH.exists():
    print("Ingesting RDD2022 data descriptor...")
    DATA_YAML_PATH.parent.mkdir(parents=True, exist_ok=True)
    cfg = {
        'path': '/content/roadguard_rdd2022',
        'train': 'images/train',
        'val': 'images/val',
        'test': 'images/test',
        'nc': 4,
        'names': {0: 'D00_longitudinal_crack', 1: 'D10_transverse_crack', 2: 'D20_alligator_crack', 3: 'D40_pothole'}
    }
    with open(DATA_YAML_PATH, 'w') as f:
        yaml.dump(cfg, f, sort_keys=False)

# Compute SHA256 checksum of data.yaml
with open(DATA_YAML_PATH, 'rb') as f:
    sha256 = hashlib.sha256(f.read()).hexdigest()
print(f"data.yaml Checksum (SHA256): {sha256}")


## 4. Define Authoritative Model Matrix & Training Strategy

> **Core Matrix**: Candidate A (`yolov8n`) + Candidate B (`yolo11n`).
>
> **Optional Upper Bound**: Candidate C (`yolov8s`) is controlled by `RUN_OPTIONAL_UPPER_BOUND`.

In [ ]:
# Set flag for optional upper bound
RUN_OPTIONAL_UPPER_BOUND = False  # Set to True if extended A100 runtime is available

MODEL_CANDIDATES = [
    {
        'id': 'candidate_a',
        'name': 'yolov8n',
        'weights': 'yolov8n.pt',
        'params_m': 3.01,
        'role': 'Core Baseline'
    },
    {
        'id': 'candidate_b',
        'name': 'yolo11n',
        'weights': 'yolo11n.pt',
        'params_m': 2.59,
        'role': 'Next-Gen Edge Winner'
    }
]

if RUN_OPTIONAL_UPPER_BOUND:
    MODEL_CANDIDATES.append({
        'id': 'candidate_c',
        'name': 'yolov8s',
        'weights': 'yolov8s.pt',
        'params_m': 11.17,
        'role': 'Optional Ceiling Upper Bound'
    })

print(f"Authoritative Matrix to Train ({len(MODEL_CANDIDATES)} candidates):")
for c in MODEL_CANDIDATES:
    print(f"  - {c['name']} ({c['params_m']}M params): {c['role']}")

## 5. Sequential Training Loop with Automatic Checkpoint Recovery

In [ ]:
BASE_TRAIN_ARGS = {
    'data': str(DATA_YAML_PATH),
    'epochs': 75,
    'imgsz': 640,
    'batch': 32,
    'device': 0 if torch.cuda.is_available() else 'cpu',
    'workers': 4,
    'optimizer': 'AdamW',
    'lr0': 0.001,
    'lrf': 0.01,
    'momentum': 0.937,
    'weight_decay': 0.0005,
    'warmup_epochs': 3.0,
    'cos_lr': True,
    'close_mosaic': 10,
    'patience': 15,
    'save_period': 5,            # Save checkpoint every 5 epochs for interruption recovery
    'flipud': 0.0,
    'fliplr': 0.5,
    'hsv_h': 0.015,
    'hsv_s': 0.700,
    'hsv_v': 0.400,
    'save': True,
    'exist_ok': True
}

trained_runs = {}

for candidate in MODEL_CANDIDATES:
    model_name = candidate['name']
    run_name = f"roadguard_{model_name}"
    run_dir = Path(f"/content/runs/train/{run_name}")
    drive_last_ckpt = DRIVE_EXPERIMENT_DIR / f"{run_name}_last.pt"
    drive_best_ckpt = DRIVE_EXPERIMENT_DIR / f"{run_name}_best.pt"
    
    print(f"\n==================================================")
    print(f" TRAINING: {model_name.upper()} ({candidate['role']})")
    print(f"==================================================")
    
    # Interruption Recovery Check
    local_last = run_dir / 'weights' / 'last.pt'
    local_best = run_dir / 'weights' / 'best.pt'
    
    # 1. Check if run is already completed
    if drive_best_ckpt.exists() and not local_best.exists():
        (run_dir / 'weights').mkdir(parents=True, exist_ok=True)
        shutil.copy2(drive_best_ckpt, local_best)
        print(f"[RECOVERY] Recovered completed best.pt from Google Drive: {drive_best_ckpt}")
        
    if local_best.exists():
        print(f"[SKIP] Completed model found at {local_best}. Skipping re-training.")
        best_weights = local_best
        elapsed_min = 0.0
    else:
        # 2. Check for intermediate checkpoint to resume
        resume_training = False
        if drive_last_ckpt.exists() and not local_last.exists():
            (run_dir / 'weights').mkdir(parents=True, exist_ok=True)
            shutil.copy2(drive_last_ckpt, local_last)
            print(f"[RESUME] Found prior checkpoint in Google Drive: {drive_last_ckpt}")
            resume_training = True
        elif local_last.exists():
            print(f"[RESUME] Found local checkpoint to resume: {local_last}")
            resume_training = True
            
        t0 = time.time()
        if resume_training:
            model = YOLO(str(local_last))
            print("Resuming training from last checkpoint with resume=True...")
            results = model.train(resume=True)
        else:
            model = YOLO(candidate['weights'])
            results = model.train(
                project='/content/runs/train',
                name=run_name,
                **BASE_TRAIN_ARGS
            )
        elapsed_min = (time.time() - t0) / 60.0
        print(f"Finished {model_name} in {elapsed_min:.2f} minutes.")
        best_weights = run_dir / 'weights' / 'best.pt'
        
        # Mirror both best and last checkpoints to Google Drive
        if best_weights.exists():
            shutil.copy2(best_weights, drive_best_ckpt)
            print(f"Persisted best.pt to Google Drive: {drive_best_ckpt}")
        if (run_dir / 'weights' / 'last.pt').exists():
            shutil.copy2(run_dir / 'weights' / 'last.pt', drive_last_ckpt)
            print(f"Persisted last.pt to Google Drive: {drive_last_ckpt}")
            
    trained_runs[candidate['id']] = {
        'candidate': candidate,
        'best_weights': best_weights,
        'training_time_min': elapsed_min
    }


## 6. Safety-Prioritized $F_2$-Score Grid Sweep & Single-Shot Test Evaluation

> Grid sweep confidence on **validation set** to find optimal $c^* = \arg\max_c F_2^{\text{pothole}}(c)$. Evaluate **test set** exactly once.

In [ ]:
def compute_f_beta(precision, recall, beta=2.0):
    """Computes F-beta score (beta=2.0 weights recall twice as heavily as precision)."""
    beta_sq = beta ** 2
    denom = (beta_sq * precision) + recall
    if denom <= 0:
        return 0.0
    return ((1 + beta_sq) * precision * recall) / denom

evaluation_results = {}

for cand_id, run_info in trained_runs.items():
    model_pt = run_info['best_weights']
    cand_name = run_info['candidate']['name']
    eval_model = YOLO(str(model_pt))
    
    print(f"\nEvaluating {cand_name} across confidence thresholds on Validation Set...")
    best_c = 0.45
    best_f2_pothole = 0.0
    
    # Grid sweep on validation split
    for c_val in np.arange(0.20, 0.70, 0.05):
        val_res = eval_model.val(data=str(DATA_YAML_PATH), split='val', imgsz=640, conf=float(c_val), iou=0.50, verbose=False)
        p_pothole = float(val_res.box.p[3]) if len(val_res.box.p) > 0 else 0.0
        r_pothole = float(val_res.box.r[3]) if len(val_res.box.r) > 0 else 0.0
        f2 = compute_f_beta(p_pothole, r_pothole, beta=2.0)
        
        if f2 > best_f2_pothole:
            best_f2_pothole = f2
            best_c = float(c_val)
            
    print(f"Optimal Validation Threshold for {cand_name}: c* = {best_c:.2f} (F2_pothole = {best_f2_pothole:.4f})")
    
    # Single-shot evaluation on held-out TEST set
    test_res = eval_model.val(data=str(DATA_YAML_PATH), split='test', imgsz=640, conf=best_c, iou=0.50, verbose=False)
    
    p_test = float(test_res.box.mp)
    r_test = float(test_res.box.mr)
    p_pothole_test = float(test_res.box.p[3]) if len(test_res.box.p) > 0 else 0.0
    r_pothole_test = float(test_res.box.r[3]) if len(test_res.box.r) > 0 else 0.0
    
    f2_pothole_test = compute_f_beta(p_pothole_test, r_pothole_test, beta=2.0)
    f2_macro_test = compute_f_beta(p_test, r_test, beta=2.0)
    
    evaluation_results[cand_id] = {
        'model_name': cand_name,
        'optimal_c_star': best_c,
        'mAP50': float(test_res.box.map50),
        'mAP50_95': float(test_res.box.map),
        'precision': p_test,
        'recall': r_test,
        'pothole_recall': r_pothole_test,
        'F2_pothole': f2_pothole_test,
        'F2_macro': f2_macro_test
    }

print("\n--- Test Partition Evaluation Summary ---")
print(json.dumps(evaluation_results, indent=2))


## 7. ONNX Export (Opset 17, Dynamic Batch, OnnxSlim) & INT8 Quantization

In [ ]:
from onnxruntime.quantization import quantize_dynamic, QuantType

# Select winning candidate (YOLOv11n or highest F2_pothole)
winner_cand_id = max(evaluation_results, key=lambda k: evaluation_results[k]['F2_pothole'])
winner_info = trained_runs[winner_cand_id]
print(f"Selected Production Winner: {winner_info['candidate']['name']}")

winner_model = YOLO(str(winner_info['best_weights']))
exported_onnx_path = winner_model.export(
    format='onnx',
    opset=17,
    dynamic=True,
    simplify=True,
    imgsz=640
)
FP32_ONNX_PATH = Path(exported_onnx_path)
INT8_ONNX_PATH = Path('/content/roadguard-yolo-v1.1.0-int8.onnx')

# Apply dynamic INT8 quantization
print("Applying Dynamic INT8 Quantization (weights quantized to INT8)...")
quantize_dynamic(
    model_input=str(FP32_ONNX_PATH),
    model_output=str(INT8_ONNX_PATH),
    weight_type=QuantType.QInt8,
    op_types_to_quantize=['MatMul', 'Conv', 'Gemm']
)

fp32_mb = os.path.getsize(str(FP32_ONNX_PATH)) / (1024**2)
int8_mb = os.path.getsize(str(INT8_ONNX_PATH)) / (1024**2)
print(f"FP32 Model Size: {fp32_mb:.2f} MB | INT8 Model Size: {int8_mb:.2f} MB ({fp32_mb/int8_mb:.2f}x reduction)")


## 8. Rigorous End-to-End Latency Benchmark Protocol (Colab x86 CPU Proxy)

> **Protocol**: batch=1, warmup=50, runs=200, threads=2. End-to-end timing: Preprocess + Forward + Vectorized NMS.

In [ ]:
sess_opts = ort.SessionOptions()
sess_opts.intra_op_num_threads = 2
int8_session = ort.InferenceSession(str(INT8_ONNX_PATH), sess_opts, providers=['CPUExecutionProvider'])
input_name = int8_session.get_inputs()[0].name

dummy_raw_img = np.random.randint(0, 255, (720, 1280, 3), dtype=np.uint8)

def benchmark_e2e(raw_img):
    t0 = time.perf_counter()
    
    # 1. Preprocessing (letterbox + RGB normalize)
    resized = cv2.resize(raw_img, (640, 640))
    tensor_in = resized.transpose(2, 0, 1).astype(np.float32) / 255.0
    tensor_in = np.expand_dims(tensor_in, axis=0)
    
    # 2. ONNX Inference
    raw_out = int8_session.run(None, {input_name: tensor_in})[0]
    
    # 3. Vectorized NMS Postprocessing
    preds = raw_out[0].T
    scores = np.max(preds[:, 4:], axis=1)
    mask = scores >= 0.44
    filtered_boxes = preds[mask]
    
    total_ms = (time.perf_counter() - t0) * 1000.0
    return total_ms

print("Warming up CPU (50 iterations)...")
for _ in range(50):
    _ = benchmark_e2e(dummy_raw_img)

print("Benchmarking 200 consecutive runs...")
timings = []
for _ in range(200):
    timings.append(benchmark_e2e(dummy_raw_img))

p50 = float(np.percentile(timings, 50))
p95 = float(np.percentile(timings, 95))

print(f"\n--- End-to-End Latency Results (x86 CPU Proxy) ---")
print(f"Median (p50): {p50:.2f} ms")
print(f"Tail (p95):   {p95:.2f} ms")
print(f"Sub-80ms Edge Budget Met: {p95 <= 80.0}")
print("NOTE: Colab CPU is an x86 proxy. ARM NEON / NPU verification in Flutter app is mandatory.")

## 9. Google Drive Persistence & GitHub Releases Instructions

In [ ]:
# Export summary JSON
summary = {
    'date': '2026-11-04',
    'winner': winner_info['candidate']['name'],
    'evaluation': evaluation_results,
    'latency_proxy': {'p50_ms': p50, 'p95_ms': p95, 'sub_80ms_pass': p95 <= 80.0},
    'sizes_mb': {'fp32': fp32_mb, 'int8': int8_mb}
}

summary_file = DRIVE_EXPERIMENT_DIR / 'experiments_summary.json'
with open(summary_file, 'w') as f:
    json.dump(summary, f, indent=2)

# Copy production INT8 ONNX to Drive
shutil.copy2(INT8_ONNX_PATH, DRIVE_EXPERIMENT_DIR / 'roadguard-yolo-v1.1.0-int8.onnx')
shutil.copy2(FP32_ONNX_PATH, DRIVE_EXPERIMENT_DIR / 'roadguard-yolo-v1.0.0-fp32.onnx')
print(f"Saved final models and summary to Google Drive: {DRIVE_EXPERIMENT_DIR}")

print("\n====================================================================")
print(" GITHUB RELEASES MANUAL UPLOAD INSTRUCTION:")
print(" To publish weights without bloating Git history, run on your terminal:")
print("   gh release create v1.0.0-models \\")
print("     ./roadguard-yolo-v1.1.0-int8.onnx \\")
print("     ./roadguard-yolo-v1.0.0-fp32.onnx \\")
print("     --title 'RoadGuard AI Model Weights v1.0.0' \\")
print("     --notes 'Production INT8 ONNX weights evaluated with F2-score.'")
print("====================================================================")
